# Macro Feature Neural Models

Trains the LSTM and graph neural models on the separate `stock_features_plus_regime_v1` dataset. HAR baselines remain unchanged.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

import config
from src.macro_dataset import load_feature_tensor, MACRO_FEATURE_VERSION
from src.models import LSTMModel, GNNModel, GNNModelV2
from src.train import (
    set_seeds, train_lstm, predict_lstm_split, train_gnn, predict_gnn_split,
)
from src.graphs import load_corr_graphs
from src.experiment_registry import register_experiment

results_dir = Path(config.DATA_RESULTS_DIR)
ckpt_dir = Path(config.CHECKPOINTS_DIR)
features_dir = Path(config.DATA_FEATURES_DIR)
graphs_dir = Path(config.DATA_GRAPHS_DIR)
results_dir.mkdir(parents=True, exist_ok=True)
ckpt_dir.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


In [2]:
# Load macro feature tensor and aligned target/splits
target = pd.read_parquet(features_dir / 'target.parquet')
target.index = pd.to_datetime(target.index)
splits = pd.read_parquet(features_dir / 'splits.parquet')
splits['week'] = pd.to_datetime(splits['week'])

features_3d, feature_names, tickers, week_index = load_feature_tensor(
    feature_path=features_dir / 'features_macro.parquet',
    meta_path=features_dir / 'features_macro_meta.json',
    target_index=target.index,
)
target = target.loc[week_index, tickers]
target_arr = target.to_numpy(dtype=float)

train_pos = np.flatnonzero(week_index.isin(splits.loc[splits['split'].eq('train'), 'week']))
val_pos = np.flatnonzero(week_index.isin(splits.loc[splits['split'].eq('val'), 'week']))
test_pos = np.flatnonzero(week_index.isin(splits.loc[splits['split'].eq('test'), 'week']))

n_weeks, n_stocks, n_feats = features_3d.shape
print('Macro tensor:', features_3d.shape)
print('Feature version:', MACRO_FEATURE_VERSION)
print('Train/val/test positions:', len(train_pos), len(val_pos), len(test_pos))
print('Last five features:', feature_names[-5:])

Macro tensor: (572, 460, 19)
Feature version: stock_features_plus_regime_v1
Train/val/test positions: 417 52 103
Last five features: ['spy_return_1m', 'treasury_10y_2y_spread', 'ig_credit_spread', 'avg_pairwise_stock_correlation', 'correlation_graph_density']


In [3]:
# Graph loaders
zero_edges = torch.zeros((2, 0), dtype=torch.long)

corr_graphs = {}
for split_name in ('train', 'val', 'test'):
    corr_graphs.update(load_corr_graphs(config.CORR_THRESHOLD, split_name))

def corr_edge_index(week):
    return corr_graphs.get(pd.Timestamp(week), zero_edges)

sector_edges = pd.read_parquet(graphs_dir / 'sector_edges_by_year.parquet')
sector_graphs = {}
for year, group in sector_edges.groupby('year'):
    sector_graphs[int(year)] = torch.tensor(group[['src', 'dst']].to_numpy(dtype=np.int64).T, dtype=torch.long)

def sector_edge_index(week):
    return sector_graphs[int(pd.Timestamp(week).year)]

granger_edges = pd.read_parquet(graphs_dir / 'granger_edges.parquet')
granger_graph = torch.tensor(granger_edges[['src', 'dst']].to_numpy(dtype=np.int64).T, dtype=torch.long)

def granger_edge_index(week):
    return granger_graph

print('Correlation graph weeks:', len(corr_graphs))
print('Sector graph years:', sorted(sector_graphs))
print('Granger edges:', granger_graph.shape[1])

Correlation graph weeks: 572
Sector graph years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Granger edges: 14187


In [4]:
# LSTM + Macro
set_seeds()
lstm = LSTMModel(input_size=n_feats).to(device)
lstm_ckpt = ckpt_dir / 'lstm_macro_best.pt'
lstm_loss_path = results_dir / 'lstm_macro_val_loss.json'

if lstm_ckpt.exists() and lstm_loss_path.exists():
    print('lstm_macro_best.pt found; loading saved checkpoint.')
    lstm.load_state_dict(torch.load(lstm_ckpt, map_location=device, weights_only=True))
    lstm_val_loss = json.load(open(lstm_loss_path))['val_loss']
else:
    lstm, lstm_val_loss = train_lstm(
        lstm, features_3d, target_arr, train_pos, val_pos, device, checkpoint_prefix='lstm_macro'
    )

lstm_val_preds = predict_lstm_split(lstm, features_3d, week_index, splits, tickers, device, split_name='val')
lstm_test_preds = predict_lstm_split(lstm, features_3d, week_index, splits, tickers, device, split_name='test')
lstm_val_preds.to_parquet(results_dir / 'lstm_macro_val_preds.parquet')
lstm_test_preds.to_parquet(results_dir / 'test_preds_lstm_macro.parquet')
print('LSTM + Macro best val MSE:', min(lstm_val_loss))
print('Saved LSTM macro predictions:', lstm_val_preds.shape, lstm_test_preds.shape)

/Users/shubhankar/Documents/Codex/2026-10-16/https-github-com-waderylan-sp500-gnn/work/sp500-gnn/.venv/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch   1  train=0.036716  val=0.021407
Epoch   2  train=0.030975  val=0.020115
Epoch   3  train=0.028396  val=0.019996
Epoch   4  train=0.027496  val=0.020039
Epoch   5  train=0.025862  val=0.020008
Epoch   6  train=0.026773  val=0.020112
Epoch   7  train=0.026154  val=0.019908
Epoch   8  train=0.026325  val=0.019977
Epoch   9  train=0.024487  val=0.019787
Epoch  10  train=0.025762  val=0.020214
Epoch  11  train=0.024260  val=0.019777
Epoch  12  train=0.024976  val=0.020078
Epoch  13  train=0.023906  val=0.019903
Epoch  14  train=0.025494  val=0.020014
Epoch  15  train=0.024075  val=0.019916
Epoch  16  train=0.025382  val=0.020100
Epoch  17  train=0.023629  val=0.019805
Epoch  18  train=0.023186  val=0.019882
Epoch  19  train=0.022626  val=0.020158
Epoch  20  train=0.022294  val=0.020060
Epoch  21  train=0.022418  val=0.020315
Early stop at epoch 21 (no improvement for 10 epochs)


/Users/shubhankar/Documents/Codex/2026-10-16/https-github-com-waderylan-sp500-gnn/work/sp500-gnn/src/train.py:240: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_s

LSTM + Macro best val MSE: 0.019776520426743306
Saved LSTM macro predictions: (52, 460) (103, 460)


In [5]:
# GNN-Correlation + Macro
set_seeds()
gnn_corr = GNNModelV2(
    in_channels=n_feats,
    hidden_dim=config.HIDDEN_DIM,
    dropout=config.DROPOUT,
    num_layers=config.GNN_NUM_LAYERS,
    batch_norm=False,
).to(device)

if (ckpt_dir / 'gnn_corr_macro_best.pt').exists() and (results_dir / 'gnn_corr_macro_val_loss.json').exists():
    print('gnn_corr_macro_best.pt found; loading saved checkpoint.')
    gnn_corr.load_state_dict(torch.load(ckpt_dir / 'gnn_corr_macro_best.pt', map_location=device, weights_only=True))
    corr_val_loss = json.load(open(results_dir / 'gnn_corr_macro_val_loss.json'))['val_loss']
else:
    gnn_corr, corr_val_loss = train_gnn(
        gnn_corr, features_3d, target_arr, week_index, corr_edge_index, splits,
        graph_type='corr_macro', device=device, max_epochs=config.GNN_MAX_EPOCHS,
        lr=config.GNN_LEARNING_RATE, patience=config.EARLY_STOP_PATIENCE, save_periodic=False,
    )

gnn_corr_val = predict_gnn_split(gnn_corr, features_3d, week_index, corr_edge_index, splits, tickers, device, split_name='val')
gnn_corr_test = predict_gnn_split(gnn_corr, features_3d, week_index, corr_edge_index, splits, tickers, device, split_name='test')
gnn_corr_val.to_parquet(results_dir / 'gnn_corr_macro_val_preds.parquet')
gnn_corr_test.to_parquet(results_dir / 'test_preds_gnn_corr_macro.parquet')
print('GNN-Correlation + Macro best val MSE:', min(corr_val_loss))

/Users/shubhankar/Documents/Codex/2026-10-16/https-github-com-waderylan-sp500-gnn/work/sp500-gnn/.venv/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/Users/shubhankar/Documents/Codex/2026-10-16/https-github-com-waderylan-sp500-gnn/work/sp500-gnn/src/models.py:350: UserWarning: GNNModelV2.forward: input features contain NaN values. Imputing with 0 (z-score mean). Expected during warm-up weeks.
  warnings.warn(


Epoch   1  train=0.035778  val=0.022737
Epoch   2  train=0.031788  val=0.021063
Epoch   3  train=0.032009  val=0.022007
Epoch   4  train=0.029278  val=0.020904
Epoch   5  train=0.030174  val=0.022557
Epoch   6  train=0.031156  val=0.021553
Epoch   7  train=0.031027  val=0.024404
Epoch   8  train=0.030190  val=0.020860
Epoch   9  train=0.030798  val=0.021560
Epoch  10  train=0.030389  val=0.022900
Epoch  11  train=0.030362  val=0.021119
Epoch  12  train=0.031946  val=0.022297
Epoch  13  train=0.040417  val=0.021693
Epoch  14  train=0.032365  val=0.020983
Epoch  15  train=0.028848  val=0.021043
Epoch  16  train=0.027248  val=0.020602
Epoch  17  train=0.027901  val=0.020799
Epoch  18  train=0.027456  val=0.020685
Epoch  19  train=0.032040  val=0.020917
Epoch  20  train=0.029041  val=0.020815
Epoch  21  train=0.027550  val=0.020534
Epoch  22  train=0.028604  val=0.020602
Epoch  23  train=0.027325  val=0.020677
Epoch  24  train=0.029800  val=0.020620
Epoch  25  train=0.027350  val=0.020466


In [6]:
# GNN-Sector + Macro
set_seeds()
gnn_sector = GNNModel(in_channels=n_feats).to(device)

if (ckpt_dir / 'gnn_sector_macro_best.pt').exists() and (results_dir / 'gnn_sector_macro_val_loss.json').exists():
    print('gnn_sector_macro_best.pt found; loading saved checkpoint.')
    gnn_sector.load_state_dict(torch.load(ckpt_dir / 'gnn_sector_macro_best.pt', map_location=device, weights_only=True))
    sector_val_loss = json.load(open(results_dir / 'gnn_sector_macro_val_loss.json'))['val_loss']
else:
    gnn_sector, sector_val_loss = train_gnn(
        gnn_sector, features_3d, target_arr, week_index, sector_edge_index, splits,
        graph_type='sector_macro', device=device, max_epochs=config.GNN_MAX_EPOCHS,
        lr=config.LEARNING_RATE, patience=config.EARLY_STOP_PATIENCE, save_periodic=False,
    )

gnn_sector_val = predict_gnn_split(gnn_sector, features_3d, week_index, sector_edge_index, splits, tickers, device, split_name='val')
gnn_sector_test = predict_gnn_split(gnn_sector, features_3d, week_index, sector_edge_index, splits, tickers, device, split_name='test')
gnn_sector_val.to_parquet(results_dir / 'gnn_sector_macro_val_preds.parquet')
gnn_sector_test.to_parquet(results_dir / 'test_preds_gnn_sector_macro.parquet')
print('GNN-Sector + Macro best val MSE:', min(sector_val_loss))

/Users/shubhankar/Documents/Codex/2026-10-16/https-github-com-waderylan-sp500-gnn/work/sp500-gnn/.venv/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/Users/shubhankar/Documents/Codex/2026-10-16/https-github-com-waderylan-sp500-gnn/work/sp500-gnn/src/models.py:289: UserWarning: GNNModel.forward: input features contain NaN values. Imputing with 0 (z-score mean). Expected during warm-up weeks.
  warnings.warn(


Epoch   1  train=0.040480  val=0.024236
Epoch   2  train=0.031488  val=0.020359
Epoch   3  train=0.032179  val=0.020500
Epoch   4  train=0.029636  val=0.020392
Epoch   5  train=0.028880  val=0.020316
Epoch   6  train=0.027532  val=0.020532
Epoch   7  train=0.032112  val=0.021911
Epoch   8  train=0.032312  val=0.022319
Epoch   9  train=0.030599  val=0.020945
Epoch  10  train=0.027861  val=0.020674
Epoch  11  train=0.028946  val=0.021090
Epoch  12  train=0.028224  val=0.020149
Epoch  13  train=0.026837  val=0.020618
Epoch  14  train=0.025009  val=0.020053
Epoch  15  train=0.025662  val=0.020365
Epoch  16  train=0.025687  val=0.021062
Epoch  17  train=0.028967  val=0.020283
Epoch  18  train=0.025593  val=0.020219
Epoch  19  train=0.023930  val=0.020009
Epoch  20  train=0.023574  val=0.020082
Epoch  21  train=0.023552  val=0.020089
Epoch  22  train=0.023708  val=0.020380
Epoch  23  train=0.024395  val=0.021472
Epoch  24  train=0.028697  val=0.021018
Epoch  25  train=0.024721  val=0.020114


In [7]:
# GNN-Granger + Macro
set_seeds()
gnn_granger = GNNModel(in_channels=n_feats).to(device)

if (ckpt_dir / 'gnn_granger_macro_best.pt').exists() and (results_dir / 'gnn_granger_macro_val_loss.json').exists():
    print('gnn_granger_macro_best.pt found; loading saved checkpoint.')
    gnn_granger.load_state_dict(torch.load(ckpt_dir / 'gnn_granger_macro_best.pt', map_location=device, weights_only=True))
    granger_val_loss = json.load(open(results_dir / 'gnn_granger_macro_val_loss.json'))['val_loss']
else:
    gnn_granger, granger_val_loss = train_gnn(
        gnn_granger, features_3d, target_arr, week_index, granger_edge_index, splits,
        graph_type='granger_macro', device=device, max_epochs=config.GNN_MAX_EPOCHS,
        lr=config.LEARNING_RATE, patience=config.EARLY_STOP_PATIENCE, save_periodic=False,
    )

gnn_granger_val = predict_gnn_split(gnn_granger, features_3d, week_index, granger_edge_index, splits, tickers, device, split_name='val')
gnn_granger_test = predict_gnn_split(gnn_granger, features_3d, week_index, granger_edge_index, splits, tickers, device, split_name='test')
gnn_granger_val.to_parquet(results_dir / 'gnn_granger_macro_val_preds.parquet')
gnn_granger_test.to_parquet(results_dir / 'test_preds_gnn_granger_macro.parquet')
print('GNN-Granger + Macro best val MSE:', min(granger_val_loss))

/Users/shubhankar/Documents/Codex/2026-10-16/https-github-com-waderylan-sp500-gnn/work/sp500-gnn/.venv/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/Users/shubhankar/Documents/Codex/2026-10-16/https-github-com-waderylan-sp500-gnn/work/sp500-gnn/src/models.py:289: UserWarning: GNNModel.forward: input features contain NaN values. Imputing with 0 (z-score mean). Expected during warm-up weeks.
  warnings.warn(


Epoch   1  train=0.039856  val=0.023369
Epoch   2  train=0.032268  val=0.020211
Epoch   3  train=0.032721  val=0.021109
Epoch   4  train=0.029782  val=0.020630
Epoch   5  train=0.028603  val=0.020347
Epoch   6  train=0.027489  val=0.020534
Epoch   7  train=0.026753  val=0.020260
Epoch   8  train=0.026194  val=0.020272
Epoch   9  train=0.029595  val=0.020312
Epoch  10  train=0.026870  val=0.020519
Epoch  11  train=0.024819  val=0.020089
Epoch  12  train=0.024539  val=0.019995
Epoch  13  train=0.027373  val=0.020295
Epoch  14  train=0.031407  val=0.020406
Epoch  15  train=0.028082  val=0.020369
Epoch  16  train=0.026277  val=0.020020
Epoch  17  train=0.026019  val=0.020231
Epoch  18  train=0.024799  val=0.019992
Epoch  19  train=0.025646  val=0.020094
Epoch  20  train=0.025087  val=0.020366
Epoch  21  train=0.028499  val=0.020141
Epoch  22  train=0.024415  val=0.020052
Epoch  23  train=0.023236  val=0.019972
Epoch  24  train=0.023288  val=0.020130
Epoch  25  train=0.026014  val=0.020133


In [8]:
# GNN-Ensemble + Macro
gnn_ensemble_val = (gnn_corr_val + gnn_sector_val + gnn_granger_val) / 3.0
gnn_ensemble_test = (gnn_corr_test + gnn_sector_test + gnn_granger_test) / 3.0
gnn_ensemble_val.to_parquet(results_dir / 'gnn_ensemble_macro_val_preds.parquet')
gnn_ensemble_test.to_parquet(results_dir / 'test_preds_gnn_ensemble_macro.parquet')
print('Saved GNN-Ensemble + Macro predictions:', gnn_ensemble_val.shape, gnn_ensemble_test.shape)

Saved GNN-Ensemble + Macro predictions: (52, 460) (103, 460)


In [9]:
# Register macro experiments
def _json(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'))

base_hparams = {
    'random_seed': config.RANDOM_SEED,
    'data_start': config.DATA_START,
    'data_end': config.DATA_END,
    'feature_artifact': 'data/features/features_macro.parquet',
    'feature_meta': 'data/features/features_macro_meta.json',
}
common = {
    'feature_version': MACRO_FEATURE_VERSION,
    'train_split': f'{config.DATA_START} through {config.TRAIN_END}',
    'val_split': f'{config.TRAIN_END} exclusive through {config.VAL_END}',
    'test_split': f'{config.VAL_END} exclusive through {config.TEST_END}',
    'test_metrics_path': '',
    'portfolio_metrics_path': '',
}
rows = [
    {**common, 'experiment_id': 'macro_lstm', 'model_name': 'LSTM + Macro', 'model_family': 'LSTM', 'graph_type': 'none', 'loss_type': 'mse', 'graph_version': 'none', 'checkpoint_path': 'data/results/checkpoints/lstm_macro_best.pt', 'hyperparameters': _json({**base_hparams, 'hidden_dim': config.LSTM_HIDDEN_DIM, 'seq_len': config.LSTM_SEQ_LEN, 'learning_rate': config.LEARNING_RATE}), 'validation_metrics_path': 'data/results/lstm_macro_val_loss.json', 'prediction_path': 'data/results/test_preds_lstm_macro.parquet', 'validation_prediction_path': 'data/results/lstm_macro_val_preds.parquet', 'notes': 'Macro/regime feature LSTM trained on stock_features_plus_regime_v1.'},
    {**common, 'experiment_id': 'macro_gnn_correlation', 'model_name': 'GNN-Correlation + Macro', 'model_family': 'GNN', 'graph_type': 'correlation', 'loss_type': 'mse', 'graph_version': f'correlation_threshold_{config.CORR_THRESHOLD}_lookback_{config.CORR_LOOKBACK_DAYS}', 'checkpoint_path': 'data/results/checkpoints/gnn_corr_macro_best.pt', 'hyperparameters': _json({**base_hparams, 'architecture': 'GNNModelV2', 'hidden_dim': config.HIDDEN_DIM, 'num_layers': config.GNN_NUM_LAYERS, 'dropout': config.DROPOUT, 'learning_rate': config.GNN_LEARNING_RATE}), 'validation_metrics_path': 'data/results/gnn_corr_macro_val_loss.json', 'prediction_path': 'data/results/test_preds_gnn_corr_macro.parquet', 'validation_prediction_path': 'data/results/gnn_corr_macro_val_preds.parquet', 'notes': 'Macro/regime feature correlation GNN.'},
    {**common, 'experiment_id': 'macro_gnn_sector', 'model_name': 'GNN-Sector + Macro', 'model_family': 'GNN', 'graph_type': 'sector', 'loss_type': 'mse', 'graph_version': 'sector_canonical_gics_labels_v1', 'checkpoint_path': 'data/results/checkpoints/gnn_sector_macro_best.pt', 'hyperparameters': _json({**base_hparams, 'architecture': 'GNNModel', 'hidden_dim': config.HIDDEN_DIM, 'dropout': config.DROPOUT, 'learning_rate': config.LEARNING_RATE}), 'validation_metrics_path': 'data/results/gnn_sector_macro_val_loss.json', 'prediction_path': 'data/results/test_preds_gnn_sector_macro.parquet', 'validation_prediction_path': 'data/results/gnn_sector_macro_val_preds.parquet', 'notes': 'Macro/regime feature sector GNN.'},
    {**common, 'experiment_id': 'macro_gnn_granger', 'model_name': 'GNN-Granger + Macro', 'model_family': 'GNN', 'graph_type': 'granger', 'loss_type': 'mse', 'graph_version': f'granger_lag_{config.GRANGER_LAG}_{config.GRANGER_CORRECTION}', 'checkpoint_path': 'data/results/checkpoints/gnn_granger_macro_best.pt', 'hyperparameters': _json({**base_hparams, 'architecture': 'GNNModel', 'hidden_dim': config.HIDDEN_DIM, 'dropout': config.DROPOUT, 'learning_rate': config.LEARNING_RATE}), 'validation_metrics_path': 'data/results/gnn_granger_macro_val_loss.json', 'prediction_path': 'data/results/test_preds_gnn_granger_macro.parquet', 'validation_prediction_path': 'data/results/gnn_granger_macro_val_preds.parquet', 'notes': 'Macro/regime feature Granger GNN.'},
    {**common, 'experiment_id': 'macro_gnn_ensemble', 'model_name': 'GNN-Ensemble + Macro', 'model_family': 'GNN ensemble', 'graph_type': 'ensemble', 'loss_type': 'mse', 'graph_version': 'correlation+sector+granger_v1', 'checkpoint_path': '', 'hyperparameters': _json({**base_hparams, 'constituents': ['macro_gnn_correlation', 'macro_gnn_sector', 'macro_gnn_granger'], 'combination': 'average_predictions'}), 'validation_metrics_path': '', 'prediction_path': 'data/results/test_preds_gnn_ensemble_macro.parquet', 'validation_prediction_path': 'data/results/gnn_ensemble_macro_val_preds.parquet', 'notes': 'Average prediction ensemble of macro GNN variants.'},
]
for row in rows:
    register_experiment(row, overwrite=True)
print('Registered macro experiments:', [row['experiment_id'] for row in rows])

Registered macro experiments: ['macro_lstm', 'macro_gnn_correlation', 'macro_gnn_sector', 'macro_gnn_granger', 'macro_gnn_ensemble']


In [10]:
# Training summary
summary = pd.DataFrame([
    {'model': 'LSTM + Macro', 'best_val_mse': min(lstm_val_loss), 'checkpoint': 'data/results/checkpoints/lstm_macro_best.pt'},
    {'model': 'GNN-Correlation + Macro', 'best_val_mse': min(corr_val_loss), 'checkpoint': 'data/results/checkpoints/gnn_corr_macro_best.pt'},
    {'model': 'GNN-Sector + Macro', 'best_val_mse': min(sector_val_loss), 'checkpoint': 'data/results/checkpoints/gnn_sector_macro_best.pt'},
    {'model': 'GNN-Granger + Macro', 'best_val_mse': min(granger_val_loss), 'checkpoint': 'data/results/checkpoints/gnn_granger_macro_best.pt'},
    {'model': 'GNN-Ensemble + Macro', 'best_val_mse': np.nan, 'checkpoint': ''},
])
display(summary)
summary.to_csv(results_dir / 'macro_model_training_summary.csv', index=False)
print('Saved data/results/macro_model_training_summary.csv')

,model,best_val_mse,checkpoint
0,LSTM + Macro,0.019777,data/results/checkpoints/lstm_macro_best.pt
1,GNN-Correlation + Macro,0.020098,data/results/checkpoints/gnn_corr_macro_best.pt
2,GNN-Sector + Macro,0.020009,data/results/checkpoints/gnn_sector_macro_best.pt
3,GNN-Granger + Macro,0.019972,data/results/checkpoints/gnn_granger_macro_bes...
4,GNN-Ensemble + Macro,NaN,


Saved data/results/macro_model_training_summary.csv
